# Prefix-Tuning

This is an example on fine-tuning Gemma with Prefix tokens. It's best to first read the [finetuning](https://gemma-llm.readthedocs.io/en/latest/finetuning.html) colab to understand this one.

See the [Prefix sampling](https://colab.research.google.com/github/google-deepmind/gemma/blob/main/colabs/prefix_sampling.ipynb) if you just want to do inference with Prefix-Tuning.


In [ ]:
!pip install -q gemma

In [ ]:
# Common imports
import os
import optax
import treescope

# Gemma imports
from kauldron import kd
from gemma import gm

By default, Jax do not utilize the full GPU memory, but this can be overwritten. See [GPU memory allocation](https://docs.jax.dev/en/latest/gpu_memory_allocation.html):

In [ ]:
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"


## Config updates

If you're familiar with the [finetuning](https://gemma-llm.readthedocs.io/en/latest/finetuning.html) tutorial, switching to Prefix-Tuning only require 3 changes to the trainer.

### 1. Model

Wrap the model in the `gm.nn.PrefixTuning.from_model`. This will apply load the learnable key-values tokens into the cache.

In [ ]:
model = gm.nn.PrefixTuning.from_model(
    prefix_length=100,
    global_layers_only=True,
    model=gm.nn.Gemma3_4B(text_only=True, tokens="batch.input"),
)

Internally, this uses the [`gemma.peft`](https://github.com/google-deepmind/gemma/blob/main/gemma/peft) mini-library to perform model surgery.

### 2. Checkpoint

Wrap the init transform in a `gm.ckpts.SkipPeft`. The wrapper is required because the param structure with and without Prefix-Tuning is different.

Only the initial pretrained weights are loaded, but the LoRA weights are kept to their random initialization.

In [ ]:
init_transform = gm.ckpts.SkipPeft(
    wrapped=gm.ckpts.LoadCheckpoint(
        path=gm.ckpts.CheckpointPath.GEMMA3_4B_IT,
    ),
)

Note: If you're loading the weights directly with `gm.ckpts.load_params`, you can use the `peft.split_params` and `peft.merge_params` instead. See [Prefix sampling](third_party/py/gemma/colabs/prefix_sampling.ipynb) for an example.

### 3. Optimizer

Finally, we add a mask to the optimizer (with `kd.optim.partial_updates`), so only the Prefix weights are trained.

In [ ]:
optimizer = kd.optim.partial_updates(
    optax.adafactor(learning_rate=1e-2),
    # We only optimize the Prefix weights. The rest of the model is frozen.
    mask=kd.optim.select("prefix_.*"),
)

## Training

### Data pipeline

Like for the [finetuning](https://gemma-llm.readthedocs.io/en/latest/finetuning.html) example, we recreate the tokenizer:

In [ ]:
tokenizer = gm.text.Gemma3Tokenizer()

tokenizer.encode('This is an example sentence', add_bos=True)

And the data pipeline:

In [ ]:
ds = kd.data.py.Tfds(
    name='mtnt/en-fr',
    split='train',
    shuffle=True,
    batch_size=8,
    transforms=[
        # Create the model inputs/targets/loss_mask.
        gm.data.Seq2SeqTask(
            # Select which field from the dataset to use.
            # https://www.tensorflow.org/datasets/catalog/mtnt
            in_prompt='src',
            in_response='dst',
            # Output batch is {'input': ..., 'target': ..., 'loss_mask': ...}
            out_input='input',
            out_target='target',
            out_target_mask='loss_mask',
            tokenizer=tokenizer,
            # Padding parameters
            max_length=200,
            truncate=True,
        ),
    ],
)

ex = ds[0]

treescope.show(ex)

We can decode an example from the batch to inspect the model input and check it is properly formatted:

In [ ]:
text = tokenizer.decode(ex['input'][0])

print(text)

### Trainer

We then create the trainer, reusing the `model`, `init_transform` and `optimizer` created above:

In [ ]:
trainer = kd.train.Trainer(
    seed=42,  # The seed of enlightenment
    workdir='/tmp/ckpts',  # TODO(epot): Make the workdir optional by default
    # Dataset
    train_ds=ds,
    # Model
    model=model,
    init_transform=init_transform,
    # Training parameters
    num_train_steps=5000,
    train_losses={
        "loss": kd.losses.SoftmaxCrossEntropyWithIntLabels(
            logits="preds.logits",
            labels="batch.target",
            mask="batch.loss_mask",
        ),
    },
    optimizer=optimizer,
)

Trainning can be launched with the `.train()` method.

Note that the trainer like the model are immutables, so it does not store the state nor params. Instead the state containing the trained parameters is returned.

In [ ]:
state, aux = trainer.train()

## Evaluation

Here, we only perform a qualitative evaluation by sampling a prompt.

For more info on evals:

* See the [sampling](https://gemma-llm.readthedocs.io/en/latest/sampling.html) tutorial for more info on running inference.
* To add evals during training, see the Kauldron [evaluator](https://kauldron.readthedocs.io/en/latest/eval.html) documentation.


In [ ]:
sampler = gm.text.ChatSampler(
    model=model,
    params=state.params,
    tokenizer=tokenizer,
)

We test a sentence, using the same formatting used during fine-tuning:

In [ ]:
sampler.chat('I\'m feeling happy today!')

We can confirm sampling works.
Note: The model predictions are bound to drift with with the addition of randomly initialized prefix tokens.